# Gold Layer - Incremental Load

### Table: gold_fact_sales

In [ ]:
MERGE INTO sales_lakehouse.dbo.gold_fact_sales AS gfs
USING (
SELECT
    sls_ord_num AS order_number,
    cu.customer_key,
    pr.product_key,
    sd.sls_order_dt AS order_date,
    sd.sls_ship_dt AS shipping_date,
    sd.sls_due_dt AS due_date,
    sd.sls_sales AS sales_amount,
    sd.sls_quantity AS quantity,
    sd.sls_price AS price
FROM sales_lakehouse.dbo.silver_crm_sales_details sd
LEFT JOIN sales_lakehouse.dbo.gold_dim_customers cu      -- get the surrogate keys from the dim tables to connect facts with dims
ON sd.sls_cust_id = cu.customer_id
LEFT JOIN sales_lakehouse.dbo.gold_dim_products pr
ON sd.sls_prd_key = pr.product_number
) t
ON gfs.order_number = t.order_number
AND gfs.product_key = t.product_key

WHEN MATCHED THEN UPDATE SET
    gfs.customer_key = t.customer_key,
    gfs.product_key = t.product_key,
    gfs.order_date = t.order_date,
    gfs.shipping_date = t.shipping_date,
    gfs.due_date = t.due_date,
    gfs.sales_amount = t.sales_amount,
    gfs.quantity = t.quantity,
    gfs.price = t.price
WHEN NOT MATCHED THEN INSERT (
    order_number,
    customer_key,
    product_key,
    order_date,
    shipping_date,
    due_date,
    sales_amount,
    quantity,
    price
)
VALUES (
    t.order_number,
    t.customer_key,
    t.product_key,
    t.order_date,
    t.shipping_date,
    t.due_date,
    t.sales_amount,
    t.quantity,
    t.price    
);

